In [1]:
!pip install mlxtend scikit-learn pandas numpy scipy


#  Imports

In [56]:
import pandas as pd
import numpy as np
import warnings
import itertools
from collections import defaultdict, Counter
from datetime import datetime

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.stats import chi2_contingency

from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

print("✅ All libraries imported successfully.")
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
print(f"Device name     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

✅ All libraries imported successfully.
PyTorch version : 2.10.0+cu128
GPU available   : True
Device name     : Tesla T4


# Load All Data




In [57]:
# ── Load all datasets ──────────────────────────────────────────────────────────
train_transactions = pd.read_csv("train_transactions.csv")
train_baskets      = pd.read_csv("train_baskets.csv")
train_users        = pd.read_csv("train_users.csv")
menu_items         = pd.read_csv("menu_items.csv")
test_instances     = pd.read_csv("test_instances.csv")
sample_submission  = pd.read_csv("sample_submission.csv")

# ── Quick shape report ─────────────────────────────────────────────────────────
for name, df in [
    ("train_transactions", train_transactions),
    ("train_baskets",      train_baskets),
    ("train_users",        train_users),
    ("menu_items",         menu_items),
    ("test_instances",     test_instances),
    ("sample_submission",  sample_submission),
]:
    print(f"{name:25s} → {df.shape[0]:>7,} rows × {df.shape[1]:>2} cols")

print("\n✅ Data loaded.")

train_transactions        → 170,793 rows × 16 cols
train_baskets             →  85,486 rows × 16 cols
train_users               →  10,000 rows ×  3 cols
menu_items                →      50 rows × 15 cols
test_instances            →  14,147 rows ×  8 cols
sample_submission         →  14,147 rows × 11 cols

✅ Data loaded.


# Preprocessing




In [58]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 – Preprocessing
# ══════════════════════════════════════════════════════════════════════════════

# ── 3.1  Parse timestamps & sort ──────────────────────────────────────────────
for df in [train_transactions, train_baskets]:
    df["timestamp"] = pd.to_datetime(df["timestamp"])

train_transactions.sort_values("timestamp", inplace=True)
train_transactions.reset_index(drop=True, inplace=True)

train_baskets.sort_values("timestamp", inplace=True)
train_baskets.reset_index(drop=True, inplace=True)

# ── 3.2  Drop duplicates ───────────────────────────────────────────────────────
train_transactions.drop_duplicates(inplace=True)
train_baskets.drop_duplicates(inplace=True)

# ── 3.3  Feature engineering on transactions ──────────────────────────────────
def engineer_features(df):
    df = df.copy()
    df["hour"]        = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.day_name()
    df["is_weekend"]  = df["timestamp"].dt.dayofweek.isin([4, 5]).astype(int)  # Fri/Sat
    df["month"]       = df["timestamp"].dt.month
    df["year"]        = df["timestamp"].dt.year

    def assign_meal_period(h):
        if 5 <= h < 12:
            return "breakfast"
        elif 12 <= h < 17:
            return "lunch"
        elif 17 <= h < 21:
            return "dinner"
        else:
            return "late_night"

    df["meal_period"] = df["hour"].apply(assign_meal_period)
    return df

train_transactions = engineer_features(train_transactions)
train_baskets      = engineer_features(train_baskets)

# ── 3.4  Parse basket items from pipe-separated strings ──────────────────────
def parse_items(raw):
    """Safely parse pipe-separated item IDs; returns a list (never NaN)."""
    if pd.isna(raw) or str(raw).strip() == "":
        return []
    return [x.strip() for x in str(raw).split("|") if x.strip() != ""]

train_baskets["item_list"] = train_baskets["item_ids"].apply(parse_items)
test_instances["basket_list"] = test_instances["current_basket_items"].apply(parse_items)

# ── 3.5  All valid item IDs from menu ─────────────────────────────────────────
ALL_ITEMS = menu_items["item_id"].tolist()
ITEM_SET  = set(ALL_ITEMS)

# ── 3.6  Recency weights on transactions ──────────────────────────────────────
# Assign a recency weight: more recent = higher weight  (exponential decay)
max_ts   = train_transactions["timestamp"].max()
min_ts   = train_transactions["timestamp"].min()
span_days = max(1, (max_ts - min_ts).days)

train_transactions["days_ago"] = (
    max_ts - train_transactions["timestamp"]
).dt.days

# decay half-life = 90 days
HALF_LIFE = 90
train_transactions["recency_weight"] = np.exp(
    -np.log(2) * train_transactions["days_ago"] / HALF_LIFE
)

# Same for baskets
train_baskets["days_ago"] = (max_ts - train_baskets["timestamp"]).dt.days
train_baskets["recency_weight"] = np.exp(
    -np.log(2) * train_baskets["days_ago"] / HALF_LIFE
)

print(f"Transactions : {len(train_transactions):,}")
print(f"Baskets      : {len(train_baskets):,}")
print(f"Date range   : {min_ts.date()} → {max_ts.date()}")
print(f"Menu items   : {len(ALL_ITEMS)}")
print("\n✅ Preprocessing complete.")

Transactions : 170,793
Baskets      : 85,486
Date range   : 2024-01-01 → 2025-07-01
Menu items   : 50

✅ Preprocessing complete.


# Train / Validation Split (Time-Based)



In [59]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 – Time-based Train / Validation Split
# ══════════════════════════════════════════════════════════════════════════════
# Strategy:
#   • Use the LAST 10 % of baskets (by timestamp) as validation.
#   • Everything before that cut-off is "seen" training data.
#   • This mimics the leaderboard: the model saw history, must predict future.

VALIDATION_FRACTION = 0.10

cutoff_idx  = int(len(train_baskets) * (1 - VALIDATION_FRACTION))
cutoff_time = train_baskets.iloc[cutoff_idx]["timestamp"]

val_baskets   = train_baskets[train_baskets["timestamp"] >= cutoff_time].copy()
train_baskets_tr = train_baskets[train_baskets["timestamp"] <  cutoff_time].copy()

# Mirror the split on transactions
val_transactions  = train_transactions[train_transactions["timestamp"] >= cutoff_time].copy()
train_tx_tr       = train_transactions[train_transactions["timestamp"] <  cutoff_time].copy()

print(f"Cut-off date         : {cutoff_time.date()}")
print(f"Train baskets        : {len(train_baskets_tr):,}")
print(f"Validation baskets   : {len(val_baskets):,}")
print(f"Train transactions   : {len(train_tx_tr):,}")
print(f"Validation tx        : {len(val_transactions):,}")

# ── Build validation instances: hide last item(s) in each val basket ──────────
# For each validation basket with ≥2 items:
#   • "context"  = all items except the last
#   • "held_out" = the last item  (ground truth for NDCG)

val_instances_list = []
for _, row in val_baskets.iterrows():
    items = row["item_list"]
    if len(items) < 2:
        continue
    context  = items[:-1]
    held_out = items[-1]
    val_instances_list.append({
        "user_id":     row["user_id"],
        "timestamp":   row["timestamp"],
        "meal_period": row["meal_period"],
        "is_weekend":  row["is_weekend"],
        "hour":        row["hour"],
        "context":     context,   # items already in basket
        "held_out":    held_out,  # what we need to predict
    })

val_df = pd.DataFrame(val_instances_list)
print(f"\nValidation prediction instances : {len(val_df):,}")
print("\n✅ Time-based split complete.")

Cut-off date         : 2025-05-08
Train baskets        : 76,937
Validation baskets   : 8,549
Train transactions   : 153,686
Validation tx        : 17,107

Validation prediction instances : 5,906

✅ Time-based split complete.


# Helper Functions

In [60]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 – Helper Functions
# ══════════════════════════════════════════════════════════════════════════════

def ndcg_at_k(recommended: list, relevant: str, k: int = 10) -> float:
    """
    Compute NDCG@k for a single instance.
    recommended : ordered list of predicted item_ids
    relevant    : single ground-truth item_id
    """
    for rank, item in enumerate(recommended[:k], start=1):
        if item == relevant:
            return 1.0 / np.log2(rank + 1)
    return 0.0


def evaluate_ndcg(predictions: dict, val_df: pd.DataFrame, k: int = 10) -> float:
    """
    Compute mean NDCG@k over all validation instances.
    predictions : {instance_key → [item_id, ...]} — key = (user_id, timestamp)
    val_df      : DataFrame with columns user_id, timestamp, held_out
    """
    scores = []
    for _, row in val_df.iterrows():
        key  = (row["user_id"], row["timestamp"])
        preds = predictions.get(key, [])
        scores.append(ndcg_at_k(preds, row["held_out"], k))
    return float(np.mean(scores))


def remove_basket_items(scores: dict, basket: list) -> dict:
    """Zero-out scores for items already in the current basket."""
    for item in basket:
        if item in scores:
            scores[item] = 0.0
    return scores


def top_k_items(scores: dict, k: int = 10, fallback: list = None) -> list:
    """
    Return top-k items by descending score.
    If fewer than k items have score > 0, pad with fallback items.
    """
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    result = [item for item, score in ranked if score > 0][:k]

    if fallback and len(result) < k:
        for item in fallback:
            if item not in result:
                result.append(item)
            if len(result) == k:
                break

    # Final safety: fill with any remaining menu item
    for item in ALL_ITEMS:
        if len(result) >= k:
            break
        if item not in result:
            result.append(item)

    return result[:k]


def normalize_scores(scores: dict) -> dict:
    """Min-max normalize a score dict to [0, 1]."""
    if not scores:
        return scores
    min_v = min(scores.values())
    max_v = max(scores.values())
    rng = max_v - min_v
    if rng == 0:
        return {k: 1.0 for k in scores}
    return {k: (v - min_v) / rng for k, v in scores.items()}


def empty_scores() -> dict:
    """Return a zeroed score dict for all menu items."""
    return {item: 0.0 for item in ALL_ITEMS}


print("✅ Helper functions defined.")

✅ Helper functions defined.


# Binary / Weighted User-Item Matrix

In [61]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 – Binary / Weighted User-Item Matrix
# ══════════════════════════════════════════════════════════════════════════════
# We build TWO matrices from train_tx_tr:
#   1) FREQUENCY matrix  : raw purchase count  per (user, item)
#   2) WEIGHTED matrix   : sum of recency_weight per (user, item)
#      → combines frequency (multiple orders → higher sum) AND recency

# ── 6.1  Aggregate ────────────────────────────────────────────────────────────
agg = (
    train_tx_tr
    .groupby(["user_id", "item_id"])
    .agg(
        freq          = ("item_id",       "count"),
        recency_sum   = ("recency_weight", "sum"),
    )
    .reset_index()
)

# ── 6.2  Pivot to matrix ──────────────────────────────────────────────────────
freq_matrix = agg.pivot(index="user_id", columns="item_id", values="freq").fillna(0)
wt_matrix   = agg.pivot(index="user_id", columns="item_id", values="recency_sum").fillna(0)

# Ensure all menu items appear as columns (add missing ones with 0)
for item in ALL_ITEMS:
    if item not in freq_matrix.columns:
        freq_matrix[item] = 0.0
    if item not in wt_matrix.columns:
        wt_matrix[item] = 0.0

freq_matrix = freq_matrix[ALL_ITEMS]
wt_matrix   = wt_matrix[ALL_ITEMS]

# ── 6.3  Binary matrix ────────────────────────────────────────────────────────
binary_matrix = (freq_matrix > 0).astype(float)

# ── 6.4  Quick stats ──────────────────────────────────────────────────────────
print(f"User-item matrix shape : {freq_matrix.shape}")
print(f"Sparsity               : {100*(1 - (freq_matrix > 0).values.mean()):.1f}%")
print(f"Max frequency cell     : {int(freq_matrix.values.max())}")
print(f"\n✅ Matrices built.")

User-item matrix shape : (8866, 50)
Sparsity               : 85.8%
Max frequency cell     : 60

✅ Matrices built.


# User Vectorization

In [62]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 – User Vectorization
# ══════════════════════════════════════════════════════════════════════════════
# Creates a rich behavioral feature vector per user using train_tx_tr.
# Features:
#   avg_price, avg_basket_size, spicy_ratio, vegetarian_ratio, vegan_ratio,
#   breakfast_ratio, lunch_ratio, dinner_ratio, late_night_ratio,
#   weekend_ratio, top_category_* (one-hot over category)

# ── Menu item lookups ─────────────────────────────────────────────────────────
item_meta = menu_items.set_index("item_id")[
    ["category", "base_price", "is_spicy", "is_vegetarian", "is_vegan"]
].to_dict(orient="index")

# ── Merge meta into transactions ──────────────────────────────────────────────
tx = train_tx_tr.copy()
tx["base_price"]    = tx["item_id"].map(lambda x: item_meta.get(x, {}).get("base_price", 0))
tx["is_spicy"]      = tx["item_id"].map(lambda x: int(item_meta.get(x, {}).get("is_spicy", False)))
tx["is_vegetarian"] = tx["item_id"].map(lambda x: int(item_meta.get(x, {}).get("is_vegetarian", False)))
tx["is_vegan"]      = tx["item_id"].map(lambda x: int(item_meta.get(x, {}).get("is_vegan", False)))
tx["category"]      = tx["item_id"].map(lambda x: item_meta.get(x, {}).get("category", "Unknown"))

ALL_CATEGORIES = menu_items["category"].unique().tolist()

def build_user_vectors(tx_df):
    """
    Return a DataFrame indexed by user_id with behavioral feature vectors.
    """
    user_records = []

    for user_id, grp in tx_df.groupby("user_id"):
        n = len(grp)

        # Price & basket
        avg_price       = grp["base_price"].mean()

        # Dietary
        spicy_ratio       = grp["is_spicy"].mean()
        vegetarian_ratio  = grp["is_vegetarian"].mean()
        vegan_ratio       = grp["is_vegan"].mean()

        # Meal period
        period_counts = grp["meal_period"].value_counts(normalize=True)
        breakfast_r   = period_counts.get("breakfast",   0.0)
        lunch_r       = period_counts.get("lunch",       0.0)
        dinner_r      = period_counts.get("dinner",      0.0)
        late_r        = period_counts.get("late_night",  0.0)

        # Weekend
        weekend_ratio = grp["is_weekend"].mean()

        # Category one-hot ratios
        cat_counts = grp["category"].value_counts(normalize=True)
        cat_feats  = {f"cat_{c}": cat_counts.get(c, 0.0) for c in ALL_CATEGORIES}

        record = {
            "user_id":          user_id,
            "avg_price":        avg_price,
            "spicy_ratio":      spicy_ratio,
            "vegetarian_ratio": vegetarian_ratio,
            "vegan_ratio":      vegan_ratio,
            "breakfast_ratio":  breakfast_r,
            "lunch_ratio":      lunch_r,
            "dinner_ratio":     dinner_r,
            "late_night_ratio": late_r,
            "weekend_ratio":    weekend_ratio,
            **cat_feats,
        }
        user_records.append(record)

    df = pd.DataFrame(user_records).set_index("user_id").fillna(0)
    return df

user_vectors = build_user_vectors(tx)

print(f"User vectors shape : {user_vectors.shape}")
print(f"Feature columns    : {list(user_vectors.columns)[:8]} ...")
print("\n✅ User vectors built.")

User vectors shape : (8866, 18)
Feature columns    : ['avg_price', 'spicy_ratio', 'vegetarian_ratio', 'vegan_ratio', 'breakfast_ratio', 'lunch_ratio', 'dinner_ratio', 'late_night_ratio'] ...

✅ User vectors built.


# Item-Item Similarity Model..





In [63]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 – Item-Item Similarity Model
# ══════════════════════════════════════════════════════════════════════════════
# We build item-item similarity using the WEIGHTED user-item matrix (transposed).
#
# Why cosine over Jaccard here?
#   • Jaccard is best for pure binary co-occurrence.
#   • Cosine respects magnitude: items bought many times by the same users
#     get stronger similarity signal — useful with recency-weighted data.
#
# item_sim[i][j] = cosine similarity between item i and item j
#                  computed over the column vectors of wt_matrix.

# ── 8.1  Item vectors = columns of wt_matrix (users × items → items × users) ─
item_vectors = wt_matrix.T.values   # shape: (n_items, n_users)

# ── 8.2  Cosine similarity ─────────────────────────────────────────────────────
item_sim_matrix = cosine_similarity(item_vectors)   # (n_items, n_items)

item_ids_ordered = wt_matrix.columns.tolist()       # same order as matrix cols
item_idx         = {item: i for i, item in enumerate(item_ids_ordered)}

def item_similarity_scores(seed_items: list, top_n: int = 30) -> dict:
    """
    Given a list of seed items (e.g., current basket),
    return a score dict {item_id: similarity_score} for all other items.
    Aggregates similarity by taking the MAX across all seeds (avoids dilution).
    """
    scores = empty_scores()
    valid_seeds = [s for s in seed_items if s in item_idx]
    if not valid_seeds:
        return scores

    for seed in valid_seeds:
        idx = item_idx[seed]
        sim_row = item_sim_matrix[idx]
        for j, item in enumerate(item_ids_ordered):
            scores[item] = max(scores.get(item, 0.0), float(sim_row[j]))

    # Remove seeds themselves
    for s in valid_seeds:
        scores[s] = 0.0

    return scores

# ── 8.3  Jaccard item-item (binary) for comparison / cross-check ──────────────
# Jaccard(A, B) = |users who bought both| / |users who bought either|
binary_vals = binary_matrix.values  # (users × items)

def jaccard_sim_matrix(bin_mat):
    """Vectorised Jaccard between all item pairs."""
    n_items = bin_mat.shape[1]
    intersection = bin_mat.T @ bin_mat                  # (n_items, n_items)
    col_sums     = bin_mat.sum(axis=0)                  # (n_items,)
    union        = col_sums[:, None] + col_sums[None, :] - intersection
    with np.errstate(divide="ignore", invalid="ignore"):
        jac = np.where(union == 0, 0.0, intersection / union)
    return jac

item_jac_matrix = jaccard_sim_matrix(binary_vals)

print(f"Cosine sim matrix shape  : {item_sim_matrix.shape}")
print(f"Jaccard sim matrix shape : {item_jac_matrix.shape}")

# ── 8.4  Sanity check: top-5 similar items for WRAP001 ───────────────────────
if "WRAP001" in item_idx:
    idx0   = item_idx["WRAP001"]
    top5   = np.argsort(item_sim_matrix[idx0])[::-1][1:6]
    print("\nTop-5 cosine-similar items to WRAP001:")
    for r in top5:
        print(f"  {item_ids_ordered[r]:12s}  sim={item_sim_matrix[idx0][r]:.4f}")

print("\n✅ Item-item similarity models built.")

Cosine sim matrix shape  : (50, 50)
Jaccard sim matrix shape : (50, 50)

Top-5 cosine-similar items to WRAP001:
  SAUCE001      sim=0.8365
  SIDE001       sim=0.4765
  WRAP004       sim=0.4386
  BEV006        sim=0.4303
  WRAP002       sim=0.4066

✅ Item-item similarity models built.


# Cell 9 - Collaborative Filtering


In [64]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 – Collaborative Filtering
# ══════════════════════════════════════════════════════════════════════════════
# Item-based CF  (primary)
#   For a user U, score(item) = Σ_{i ∈ history(U)} wt(i) × sim(i, item)
#   → weighted by the recency-weighted purchase count
#
# User-based CF  (secondary, used as a score signal only)
#   Find K most similar users → take their top items

# ── 9.1  Item-based CF ────────────────────────────────────────────────────────
def item_based_cf_scores(user_id: str, basket: list, top_n: int = 30) -> dict:
    """
    Score all items for a user using item-based collaborative filtering.
    Uses the WEIGHTED matrix (recency-aware).
    """
    scores = empty_scores()

    if user_id not in wt_matrix.index:
        # Cold-start: fall back to basket similarity only
        return item_similarity_scores(basket)

    user_row = wt_matrix.loc[user_id].values   # (n_items,)

    # Weighted sum of similarities
    for j, target_item in enumerate(item_ids_ordered):
        if target_item in ITEM_SET:
            val = float(user_row @ item_sim_matrix[:, j])
            scores[target_item] = val

    # Zero-out items the user already bought heavily (optional — keep basket filter later)
    return scores


# ── 9.2  User-based CF ────────────────────────────────────────────────────────
# Build user-user cosine similarity from user_vectors (behavioral features)
# We do this lazily: only compute for users present in user_vectors

USER_VEC_MATRIX = user_vectors.values                       # (n_users, n_feats)
USER_VEC_INDEX  = {uid: i for i, uid in enumerate(user_vectors.index)}

# Pre-compute full user-user sim matrix only if ≤ 5000 users (memory guard)
if len(user_vectors) <= 6000:
    user_sim_matrix = cosine_similarity(USER_VEC_MATRIX)    # (n_users, n_users)
    _ub_available   = True
    print(f"User-user similarity matrix built: {user_sim_matrix.shape}")
else:
    user_sim_matrix = None
    _ub_available   = False
    print("User-user matrix skipped (>6000 users) — using item-CF only.")


def user_based_cf_scores(user_id: str, k_neighbors: int = 20) -> dict:
    """
    Score items using user-based CF.
    Returns empty scores if user not in training or matrix too large.
    """
    scores = empty_scores()

    if not _ub_available or user_id not in USER_VEC_INDEX:
        return scores

    u_idx        = USER_VEC_INDEX[user_id]
    sim_row      = user_sim_matrix[u_idx]
    neighbor_idx = np.argsort(sim_row)[::-1][1 : k_neighbors + 1]

    for n_idx in neighbor_idx:
        neighbor_uid = user_vectors.index[n_idx]
        sim_val      = sim_row[n_idx]
        if sim_val <= 0 or neighbor_uid not in wt_matrix.index:
            continue
        neighbor_row = wt_matrix.loc[neighbor_uid].values
        for j, item in enumerate(item_ids_ordered):
            scores[item] += sim_val * neighbor_row[j]

    return scores


print("\n✅ Collaborative filtering functions ready.")

User-user matrix skipped (>6000 users) — using item-CF only.

✅ Collaborative filtering functions ready.


 # Cell 10 - Frequent Patterns & Association Rules ..


In [65]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 10 – Frequent Patterns & Association Rules (FP-Growth)
# ══════════════════════════════════════════════════════════════════════════════

# ── 10.1  Build transaction list from train_baskets_tr ────────────────────────
basket_transactions = train_baskets_tr["item_list"].tolist()
basket_transactions = [b for b in basket_transactions if len(b) >= 2]

print(f"Baskets for FP-Growth : {len(basket_transactions):,}")

# ── 10.2  One-hot encode with TransactionEncoder ─────────────────────────────
te  = TransactionEncoder()
te_array = te.fit_transform(basket_transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

# ── 10.3  FP-Growth (faster than Apriori on sparse data) ─────────────────────
MIN_SUPPORT = 0.005   # 0.5% of baskets — low threshold to catch rare but good pairs

print(f"Running FP-Growth with min_support={MIN_SUPPORT} ...")
frequent_itemsets = fpgrowth(basket_df, min_support=MIN_SUPPORT, use_colnames=True)
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)

print(f"Frequent itemsets found : {len(frequent_itemsets):,}")
print(frequent_itemsets["length"].value_counts().sort_index().to_string())

# ── 10.4  Association Rules ───────────────────────────────────────────────────
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0   # lift ≥ 1: consequent more likely given antecedent
)
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)

print(f"\nAssociation rules found : {len(rules):,}")
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10).to_string())

# ── 10.5  Build lookup: antecedent frozenset → list of (consequent, score) ────
# score = confidence × lift  (rewards both precision and unexpectedness)
rule_lookup = defaultdict(list)
for _, row in rules.iterrows():
    ant  = frozenset(row["antecedents"])
    cons = list(row["consequents"])
    score = float(row["confidence"] * row["lift"])
    for c in cons:
        rule_lookup[ant].append((c, score))

# Sort each list by score descending
for k in rule_lookup:
    rule_lookup[k].sort(key=lambda x: x[1], reverse=True)

print(f"\nUnique antecedent sets in lookup : {len(rule_lookup):,}")
print("\n✅ Association rules ready.")

Baskets for FP-Growth : 53,193
Running FP-Growth with min_support=0.005 ...
Frequent itemsets found : 106
length
1    32
2    58
3    16

Association rules found : 58
  antecedents consequents   support  confidence      lift
0   (HIGH002)  (SAUCE003)  0.005245    0.305586  9.352724
1  (SAUCE003)   (HIGH002)  0.005245    0.160529  9.352724
2   (WRAP004)  (SAUCE003)  0.010565    0.126150  3.860942
3  (SAUCE003)   (WRAP004)  0.010565    0.323360  3.860942
4   (WRAP002)  (SAUCE003)  0.006956    0.123498  3.779764
5  (SAUCE003)   (WRAP002)  0.006956    0.212888  3.779764
6  (PLATE002)    (BEV006)  0.009080    0.348988  2.337414
7    (BEV006)  (PLATE002)  0.009080    0.060816  2.337414
8  (PLATE004)    (BEV006)  0.008817    0.340843  2.282859
9    (BEV006)  (PLATE004)  0.008817    0.059053  2.282859

Unique antecedent sets in lookup : 22

✅ Association rules ready.


# Cell 11 - Closed & Maximal Frequent Itemsets


In [66]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 11 – Closed & Maximal Frequent Itemsets
# ══════════════════════════════════════════════════════════════════════════════
# Purpose: reduce noise in frequent_itemsets by keeping only informative ones.
#
# • CLOSED itemset  : no proper superset has the same support.
#   → Lossless compression — we keep all information.
# • MAXIMAL itemset : no proper superset is frequent at all.
#   → Aggressive pruning — useful as a compact summary.
#
# We use closed itemsets to build a cleaner association rule set,
# and maximal itemsets to understand the largest reliable co-purchase groups.

def get_closed_itemsets(fi_df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter frequent_itemsets to keep only CLOSED ones.
    An itemset X is closed if no superset of X has the same support as X.
    """
    closed = []
    rows = fi_df.to_dict(orient="records")

    for i, row_i in enumerate(rows):
        itemset_i = row_i["itemsets"]
        support_i = row_i["support"]
        is_closed = True
        for j, row_j in enumerate(rows):
            if i == j:
                continue
            itemset_j = row_j["itemsets"]
            # Check if j is a proper superset of i with the same support
            if itemset_i < itemset_j and abs(row_j["support"] - support_i) < 1e-9:
                is_closed = False
                break
        if is_closed:
            closed.append(row_i)

    return pd.DataFrame(closed).reset_index(drop=True)


def get_maximal_itemsets(fi_df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter frequent_itemsets to keep only MAXIMAL ones.
    An itemset X is maximal if no proper superset of X is in fi_df.
    """
    maximal = []
    rows = fi_df.to_dict(orient="records")
    all_itemsets = [r["itemsets"] for r in rows]

    for i, row_i in enumerate(rows):
        itemset_i = row_i["itemsets"]
        is_maximal = True
        for j, row_j in enumerate(rows):
            if i == j:
                continue
            # If a proper superset exists, i is NOT maximal
            if itemset_i < row_j["itemsets"]:
                is_maximal = False
                break
        if is_maximal:
            maximal.append(row_i)

    return pd.DataFrame(maximal).reset_index(drop=True)


# ── Run on the discovered frequent itemsets ───────────────────────────────────
closed_itemsets  = get_closed_itemsets(frequent_itemsets)
maximal_itemsets = get_maximal_itemsets(frequent_itemsets)

print(f"All frequent itemsets : {len(frequent_itemsets):,}")
print(f"Closed itemsets       : {len(closed_itemsets):,}")
print(f"Maximal itemsets      : {len(maximal_itemsets):,}")

# ── Build a closed-rule lookup (same structure as rule_lookup) ─────────────────
closed_rules = association_rules(
    closed_itemsets,
    metric="lift",
    min_threshold=1.0
)
closed_rules = closed_rules.sort_values("lift", ascending=False).reset_index(drop=True)

closed_rule_lookup = defaultdict(list)
for _, row in closed_rules.iterrows():
    ant   = frozenset(row["antecedents"])
    cons  = list(row["consequents"])
    score = float(row["confidence"] * row["lift"])
    for c in cons:
        closed_rule_lookup[ant].append((c, score))

for k in closed_rule_lookup:
    closed_rule_lookup[k].sort(key=lambda x: x[1], reverse=True)

print(f"\nClosed-rule antecedent sets : {len(closed_rule_lookup):,}")
print(f"\nTop-5 maximal itemsets:")
print(maximal_itemsets.sort_values("support", ascending=False).head(5)[["itemsets", "support"]].to_string())

print("\n✅ Closed & maximal itemsets ready.")

All frequent itemsets : 106
Closed itemsets       : 106
Maximal itemsets      : 56

Closed-rule antecedent sets : 22

Top-5 maximal itemsets:
                        itemsets   support
8    (SAUCE001, BEV006, WRAP001)  0.040644
28  (SAUCE001, SIDE001, WRAP001)  0.038088
35           (SAUCE001, WRAP002)  0.031583
51           (WRAP006, SAUCE001)  0.029985
47           (SAUCE001, WRAP003)  0.017164

✅ Closed & maximal itemsets ready.


# Cell 12 - Chi-Square & Correlation Analysis


In [67]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 12 – Chi-Square & Correlation Analysis
# ══════════════════════════════════════════════════════════════════════════════
# Goal: understand which contextual features (meal_period, is_weekend, hour_bin)
#       are statistically associated with item categories.
# Results are used to BUILD context-aware popularity weights for reranking.

tx_meta = train_tx_tr.copy()
tx_meta["category"] = tx_meta["item_id"].map(
    lambda x: item_meta.get(x, {}).get("category", "Unknown")
)
tx_meta["hour_bin"] = pd.cut(
    tx_meta["hour"],
    bins=[0, 6, 12, 17, 21, 24],
    labels=["night", "morning", "afternoon", "evening", "late"],
    right=False
)

def chi2_association(df, col_a, col_b):
    """Run chi-square test between two categorical columns."""
    ct  = pd.crosstab(df[col_a], df[col_b])
    chi2, p, dof, _ = chi2_contingency(ct)
    return chi2, p, dof, ct

print("=" * 60)
print("Chi-Square Tests: Context Features vs Item Category")
print("=" * 60)

for feat in ["meal_period", "is_weekend", "hour_bin"]:
    chi2_val, p_val, dof, ct = chi2_association(tx_meta, feat, "category")
    sig = "✅ significant" if p_val < 0.05 else "❌ not significant"
    print(f"\n{feat:15s} vs category → χ²={chi2_val:,.1f}, p={p_val:.2e}, dof={dof} — {sig}")

# ── Build context-aware popularity: P(item | meal_period) ────────────────────
# This is the key output used in the scoring system (Cell 13).

context_popularity = {}   # {meal_period: {item_id: normalized_score}}

for period, grp in tx_meta.groupby("meal_period"):
    counts = grp["item_id"].value_counts()
    total  = counts.sum()
    context_popularity[period] = (counts / total).to_dict()

# ── Also build weekend / weekday popularity ────────────────────────────────────
weekend_popularity  = {}
for is_wknd, grp in tx_meta.groupby("is_weekend"):
    counts = grp["item_id"].value_counts()
    total  = counts.sum()
    weekend_popularity[is_wknd] = (counts / total).to_dict()

# ── Global popularity (fallback) ──────────────────────────────────────────────
global_pop_counts = tx_meta["item_id"].value_counts()
global_popularity = (global_pop_counts / global_pop_counts.sum()).to_dict()

print("\n\nContext Popularity Sample (lunch, top-5):")
lunch_pop = sorted(context_popularity.get("lunch", {}).items(), key=lambda x: x[1], reverse=True)
for item, score in lunch_pop[:5]:
    print(f"  {item:12s} {score:.4f}")

# ── Correlation: hour vs unit_price ──────────────────────────────────────────
hour_price_corr = tx_meta["hour"].corr(tx_meta["unit_price"])
print(f"\nPearson correlation (hour vs unit_price) = {hour_price_corr:.4f}")

print("\n✅ Chi-square analysis & context popularity complete.")

Chi-Square Tests: Context Features vs Item Category

meal_period     vs category → χ²=3,619.6, p=0.00e+00, dof=24 — ✅ significant

is_weekend      vs category → χ²=14.3, p=7.54e-02, dof=8 — ❌ not significant

hour_bin        vs category → χ²=3,752.4, p=0.00e+00, dof=32 — ✅ significant


Context Popularity Sample (lunch, top-5):
  WRAP001      0.3610
  SAUCE001     0.2394
  SIDE001      0.0469
  BEV006       0.0446
  HIGH001      0.0418

Pearson correlation (hour vs unit_price) = -0.0072

✅ Chi-square analysis & context popularity complete.


# Cell 13 - Scoring System


In [71]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 13 – Scoring System (Updated V3)
# Improvements added:
#   - #4: Smart empty basket strategy
#   - #1: Two-stage pipeline (candidate generation → transformer reranking)
# ══════════════════════════════════════════════════════════════════════════════

# ── Weights V3 ────────────────────────────────────────────────────────────────
WEIGHTS = {
    "transformer":  0.28,   # reduced slightly — two-stage handles reranking
    "similarity":   0.25,   # strong co-purchase patterns — 0.6957
    "cf_item":      0.17,   # complements similarity — 0.6788
    "context_pop":  0.12,   # reliable fallback + context awareness
    "association":  0.10,   # useful when basket is non-empty
    "recency":      0.08,   # small honest boost for recent items
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-6, "Weights must sum to 1!"


# ── 13.1  Normalization functions ─────────────────────────────────────────────

def normalize_softmax(scores: dict, temperature: float = 0.1) -> dict:
    """
    Softmax normalization for DENSE signals (transformer, similarity, CF, popularity).
    Temperature=0.1 works well for 50-item catalogs:
      - low  temp → winner-takes-most (sharp)
      - high temp → nearly uniform (bad)
    """
    if not scores:
        return scores
    items    = list(scores.keys())
    vals     = np.array([scores[i] for i in items], dtype=np.float64)
    vals     = vals - vals.max()
    exp_vals = np.exp(vals / temperature)
    sum_exp  = exp_vals.sum()

    if sum_exp == 0:
        return {i: 1.0 / len(items) for i in items}

    return {items[k]: float(exp_vals[k] / sum_exp) for k in range(len(items))}


def normalize_minmax(scores: dict) -> dict:
    """
    Min-max normalization for SPARSE signals (association rules, recency).
    Preserves zero structure — does not inflate weak signals.
    """
    if not scores:
        return scores
    vals  = list(scores.values())
    min_v = min(vals)
    max_v = max(vals)
    rng   = max_v - min_v
    if rng == 0:
        return {k: 0.0 for k in scores}
    return {k: (v - min_v) / rng for k, v in scores.items()}


# ── 13.2  Association rule score ──────────────────────────────────────────────
def association_score(basket: list, lookup: dict = None) -> dict:
    """
    Score items using all matching association rules.
    Tries subsets of the basket from largest to smallest.
    Uses closed_rule_lookup by default (less noise than full rules).
    Sparse signal → normalized with min-max.
    """
    if lookup is None:
        lookup = closed_rule_lookup

    scores     = empty_scores()
    basket_set = frozenset(basket)

    for r in range(len(basket), 0, -1):
        for subset in itertools.combinations(basket, r):
            ant = frozenset(subset)
            if ant in lookup:
                for item, score in lookup[ant]:
                    if item in scores:
                        scores[item] = max(scores[item], score)

    return scores


# ── 13.3  Recency score ───────────────────────────────────────────────────────
def recency_score_for_user(user_id: str, last_n: int = 10) -> dict:
    """
    Score based on the last N items the user purchased.
    Exponential decay: rank 1 (most recent) = 1.0, rank 2 = 0.9, ...
    Sparse signal → normalized with min-max.
    """
    scores = empty_scores()
    if user_id not in train_tx_tr["user_id"].values:
        return scores

    user_tx = (
        train_tx_tr[train_tx_tr["user_id"] == user_id]
        .sort_values("timestamp", ascending=False)
        .head(last_n)
    )

    for rank, (_, row) in enumerate(user_tx.iterrows(), start=1):
        item = row["item_id"]
        if item in scores:
            scores[item] = max(scores[item], 0.9 ** (rank - 1))

    return scores


# ── 13.4  Context popularity score ───────────────────────────────────────────
def context_pop_score(meal_period: str, is_weekend: int) -> dict:
    """
    Blend meal-period popularity (70%) and weekend popularity (30%).
    Dense signal → normalized with softmax.
    """
    scores       = empty_scores()
    period_dict  = context_popularity.get(meal_period, global_popularity)
    weekend_dict = weekend_popularity.get(is_weekend,  global_popularity)

    for item in ALL_ITEMS:
        p = period_dict.get(item, 0.0)
        w = weekend_dict.get(item, 0.0)
        scores[item] = 0.7 * p + 0.3 * w

    return scores


# ── 13.5  Improvement #4: Smart Empty Basket Strategy ────────────────────────
def smart_empty_basket_recommend(
    user_id:     str,
    meal_period: str,
    is_weekend:  int,
    k:           int = 10,
) -> list:
    """
    Special strategy when basket is completely empty.
    Transformer is unreliable with empty basket — skip it entirely.
    Uses user history + context popularity + recency instead.

    Weight logic:
      45% user history  — what does THIS user usually order?
      40% context pop   — what do others order at THIS time?
      15% recency       — what did this user order recently?
    """
    hist = normalize_softmax(user_history_score(user_id))
    ctx  = normalize_softmax(context_pop_score(meal_period, is_weekend))
    rec  = normalize_minmax(recency_score_for_user(user_id))

    final = {}
    for item in ALL_ITEMS:
        final[item] = (
            0.45 * hist.get(item, 0.0) +
            0.40 * ctx.get(item,  0.0) +
            0.15 * rec.get(item,  0.0)
        )

    ranked   = sorted(final, key=final.get, reverse=True)
    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)

    result = ranked[:k]
    for item in fallback:
        if len(result) >= k:
            break
        if item not in result:
            result.append(item)

    return result[:k]


# ── 13.6  Improvement #1: Two-Stage Pipeline ─────────────────────────────────
def two_stage_recommend(
    user_id:      str,
    basket:       list,
    meal_period:  str,
    is_weekend:   int,
    k:            int = 10,
    n_candidates: int = 20,
) -> list:
    """
    Stage 1: Generate N candidates using fast signals (sim, CF, rules, pop)
    Stage 2: Rerank candidates using transformer only (most accurate signal)

    Why this beats flat ensemble:
      - Transformer focuses only on 20 good candidates instead of all 50 items
      - Weaker signals handle candidate generation (their strength)
      - Transformer handles final ranking (its strength)
      - Less noise: transformer not penalized by random items with 0 history

    n_candidates=20 gives enough diversity while keeping transformer focused.
    """
    # ── Empty basket: dedicated strategy (transformer useless here) ───────────
    if not basket:
        return smart_empty_basket_recommend(user_id, meal_period, is_weekend, k)

    basket_set = set(basket)
    candidates = []
    seen       = set()

    def add_candidates(scored_dict: dict, top_n: int):
        """Add top_n items from a scored dict to candidates list."""
        ranked = sorted(scored_dict, key=scored_dict.get, reverse=True)
        for item in ranked[:top_n]:
            if item not in basket_set and item not in seen:
                candidates.append(item)
                seen.add(item)

    # From item similarity — strongest non-transformer signal
    sim_sc = normalize_softmax(item_similarity_scores(basket))
    add_candidates(sim_sc, 15)

    # From item-based CF — complements similarity
    cf_sc = normalize_softmax(item_based_cf_scores(user_id, basket))
    add_candidates(cf_sc, 15)

    # From association rules — basket-specific signal
    assoc_sc = normalize_minmax(association_score(basket))
    add_candidates(assoc_sc, 10)

    # From context popularity — reliable safety net
    ctx_sc = normalize_softmax(context_pop_score(meal_period, is_weekend))
    add_candidates(ctx_sc, 10)

    # Pad to n_candidates if needed
    for item in ALL_ITEMS:
        if len(candidates) >= n_candidates:
            break
        if item not in basket_set and item not in seen:
            candidates.append(item)
            seen.add(item)

    # ── Stage 2: Rerank candidates with transformer ───────────────────────────
    try:
        trans_sc = transformer_scores(basket)
        candidate_scores = {
            item: trans_sc.get(item, 0.0)
            for item in candidates
        }
        ranked = sorted(
            candidate_scores, key=candidate_scores.get, reverse=True
        )
        result = ranked[:k]

    except NameError:
        # Transformer not trained yet — fall back to combine_scores
        result = candidates[:k]

    # ── Final safety pad ──────────────────────────────────────────────────────
    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
    for item in fallback:
        if len(result) >= k:
            break
        if item not in result and item not in basket_set:
            result.append(item)

    return result[:k]


# ── 13.7  Full Ensemble (kept as backup / comparison) ────────────────────────
def combine_scores(
    user_id:     str,
    basket:      list,
    meal_period: str,
    is_weekend:  int,
    k:           int = 10,
    weights:     dict = None,
    temperature: float = 0.1,
) -> list:
    """
    Flat weighted ensemble V3.
    Kept for comparison with two_stage_recommend in Cell 14.
    Empty basket → smart_empty_basket_recommend automatically.
    """
    # ── Empty basket: dedicated strategy ──────────────────────────────────────
    if not basket:
        return smart_empty_basket_recommend(user_id, meal_period, is_weekend, k)

    if weights is None:
        weights = WEIGHTS

    basket_set = set(basket)

    # ── Step 1: Raw scores ─────────────────────────────────────────────────────
    try:
        raw_trans = transformer_scores(basket)
    except NameError:
        raw_trans = empty_scores()

    raw_sim   = item_similarity_scores(basket)
    raw_cf    = item_based_cf_scores(user_id, basket)
    raw_ctx   = context_pop_score(meal_period, is_weekend)
    raw_assoc = association_score(basket)
    raw_rec   = recency_score_for_user(user_id)

    # ── Step 2: Normalize ──────────────────────────────────────────────────────
    n_trans = normalize_softmax(raw_trans, temperature)
    n_sim   = normalize_softmax(raw_sim,   temperature)
    n_cf    = normalize_softmax(raw_cf,    temperature)
    n_ctx   = normalize_softmax(raw_ctx,   temperature)
    n_assoc = normalize_minmax(raw_assoc)
    n_rec   = normalize_minmax(raw_rec)

    # ── Step 3: Weighted combination ──────────────────────────────────────────
    final = {}
    for item in ALL_ITEMS:
        if item in basket_set:
            continue
        final[item] = (
            weights["transformer"] * n_trans.get(item, 0.0) +
            weights["similarity"]  * n_sim.get(item,   0.0) +
            weights["cf_item"]     * n_cf.get(item,    0.0) +
            weights["context_pop"] * n_ctx.get(item,   0.0) +
            weights["association"] * n_assoc.get(item, 0.0) +
            weights["recency"]     * n_rec.get(item,   0.0)
        )

    # ── Step 4: Basket co-purchase bonus ──────────────────────────────────────
    if basket:
        for r in range(len(basket), 0, -1):
            for subset in itertools.combinations(basket, r):
                ant = frozenset(subset)
                if ant in closed_rule_lookup:
                    for cons_item, rule_score in closed_rule_lookup[ant]:
                        if cons_item in final:
                            final[cons_item] += 0.03 * min(rule_score, 1.0)

    # ── Step 5: Rank ───────────────────────────────────────────────────────────
    ranked = sorted(final.items(), key=lambda x: x[1], reverse=True)
    result = [item for item, _ in ranked[:k]]

    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
    for item in fallback:
        if len(result) >= k:
            break
        if item not in result and item not in basket_set:
            result.append(item)

    return result[:k]


# ── Aliases ───────────────────────────────────────────────────────────────────
# score_items points to two_stage_recommend as the PRIMARY function
# combine_scores kept for comparison in Cell 14
score_items = two_stage_recommend

print("✅ Scoring system V3 ready.")
print(f"Weights            : { {k: round(v, 2) for k, v in WEIGHTS.items()} }")
print(f"Primary function   : two_stage_recommend (candidate gen → transformer rerank)")
print(f"Backup function    : combine_scores (flat weighted ensemble)")
print(f"Empty basket       : smart_empty_basket_recommend (history + context + recency)")
print(f"Normalization      : softmax for dense signals, minmax for sparse signals")

✅ Scoring system V3 ready.
Weights            : {'transformer': 0.28, 'similarity': 0.25, 'cf_item': 0.17, 'context_pop': 0.12, 'association': 0.1, 'recency': 0.08}
Primary function   : two_stage_recommend (candidate gen → transformer rerank)
Backup function    : combine_scores (flat weighted ensemble)
Empty basket       : smart_empty_basket_recommend (history + context + recency)
Normalization      : softmax for dense signals, minmax for sparse signals


# Cell 14 - Internal Evaluation..


In [74]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 14 – Internal Evaluation (NDCG@10) — Updated V3
# Added: Two-Stage Pipeline evaluation
# ══════════════════════════════════════════════════════════════════════════════

EVAL_SAMPLE = min(2000, len(val_df))
val_sample  = val_df.sample(EVAL_SAMPLE, random_state=42).reset_index(drop=True)

def run_eval(score_fn_name: str, score_fn) -> float:
    """
    Run a single scoring strategy over the validation sample.
    score_fn(user_id, basket, meal_period, is_weekend) → [item_id, ...]
    Returns mean NDCG@10.
    """
    preds = {}
    for _, row in val_sample.iterrows():
        key  = (row["user_id"], row["timestamp"])
        recs = score_fn(
            row["user_id"],
            row["context"],
            row["meal_period"],
            row["is_weekend"],
        )
        preds[key] = recs
    score = evaluate_ndcg(preds, val_sample)
    print(f"  {score_fn_name:35s} NDCG@10 = {score:.4f}")
    return score


print("Evaluating individual signals on validation sample ...")
print("-" * 58)


# ── 1. Context Popularity baseline ────────────────────────────────────────────
def ctx_only(user_id, basket, meal_period, is_weekend, k=10):
    s        = context_pop_score(meal_period, is_weekend)
    s        = remove_basket_items(s, basket)
    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
    return top_k_items(s, k=k, fallback=fallback)


# ── 2. Association rules only ─────────────────────────────────────────────────
def assoc_only(user_id, basket, meal_period, is_weekend, k=10):
    s        = normalize_minmax(association_score(basket))
    s        = remove_basket_items(s, basket)
    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
    return top_k_items(s, k=k, fallback=fallback)


# ── 3. Item similarity only ───────────────────────────────────────────────────
def sim_only(user_id, basket, meal_period, is_weekend, k=10):
    s        = normalize_softmax(item_similarity_scores(basket))
    s        = remove_basket_items(s, basket)
    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
    return top_k_items(s, k=k, fallback=fallback)


# ── 4. Item-based CF only ─────────────────────────────────────────────────────
def cf_only(user_id, basket, meal_period, is_weekend, k=10):
    s        = normalize_softmax(item_based_cf_scores(user_id, basket))
    s        = remove_basket_items(s, basket)
    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
    return top_k_items(s, k=k, fallback=fallback)


# ── 5. Full Ensemble V3 (flat weighted — uses combine_scores) ─────────────────
def ensemble(user_id, basket, meal_period, is_weekend, k=10):
    return combine_scores(user_id, basket, meal_period, is_weekend, k)


# ── 6. Two-Stage Pipeline ─────────────────────────────────────────────────────
# Stage 1: candidates from sim + CF + rules + popularity
# Stage 2: rerank with transformer
# Safe: if transformer not trained yet → falls back to combine_scores
def two_stage_only(user_id, basket, meal_period, is_weekend, k=10):
    return two_stage_recommend(user_id, basket, meal_period, is_weekend, k)


# ── 7. Transformer only ───────────────────────────────────────────────────────
# Safe wrapper: if Cell 14b has NOT been run yet → returns popularity fallback.
# Run Cell 14 once → run Cell 14b → re-run Cell 14 to see real score.
def transformer_only(user_id, basket, meal_period, is_weekend, k=10):
    try:
        s        = normalize_softmax(transformer_scores(basket))
        s        = remove_basket_items(s, basket)
        fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
        return top_k_items(s, k=k, fallback=fallback)
    except NameError:
        # Transformer not trained yet — return popularity as placeholder
        return ctx_only(user_id, basket, meal_period, is_weekend, k)


# ── Evaluation loop ───────────────────────────────────────────────────────────
results = {}
for name, fn in [
    ("Context Popularity (baseline)",  ctx_only),
    ("Association Rules only",         assoc_only),
    ("Item-Item Similarity only",      sim_only),
    ("Item-Based CF only",             cf_only),
    ("Full Ensemble V3",               ensemble),
    ("Two-Stage Pipeline",             two_stage_only),   # ← NEW
    ("Masked Basket Transformer",      transformer_only),
]:
    results[name] = run_eval(name, fn)

print("-" * 58)
best_method = max(results, key=results.get)
print(f"\n🏆 Best method : {best_method}  →  {results[best_method]:.4f}")

# ── Score improvement over baseline ───────────────────────────────────────────
baseline = results["Context Popularity (baseline)"]
print(f"\nImprovement over baseline:")
for name, score in results.items():
    if name == "Context Popularity (baseline)":
        continue
    delta = score - baseline
    arrow = "✅" if delta > 0 else "❌"
    print(f"  {arrow} {name:35s} {delta:+.4f}")

# ── Two-stage vs flat ensemble comparison ─────────────────────────────────────
print(f"\nTwo-Stage vs Flat Ensemble:")
ts_score  = results.get("Two-Stage Pipeline",  0)
ens_score = results.get("Full Ensemble V3",    0)
diff      = ts_score - ens_score
arrow     = "✅ Two-Stage wins" if diff > 0 else "❌ Flat Ensemble wins"
print(f"  Two-Stage Pipeline : {ts_score:.4f}")
print(f"  Full Ensemble V3   : {ens_score:.4f}")
print(f"  Difference         : {diff:+.4f}  →  {arrow}")

print("\n✅ Internal evaluation complete.")
print("\nNOTE: If Transformer shows 0.6441 (same as baseline),")
print("      run Cell 14b first, then re-run this cell.")

Evaluating individual signals on validation sample ...
----------------------------------------------------------
  Context Popularity (baseline)       NDCG@10 = 0.6441
  Association Rules only              NDCG@10 = 0.6661
  Item-Item Similarity only           NDCG@10 = 0.6957
  Item-Based CF only                  NDCG@10 = 0.6788
  Full Ensemble V3                    NDCG@10 = 0.7007
  Two-Stage Pipeline                  NDCG@10 = 0.7276
  Masked Basket Transformer           NDCG@10 = 0.7201
----------------------------------------------------------

🏆 Best method : Two-Stage Pipeline  →  0.7276

Improvement over baseline:
  ✅ Association Rules only              +0.0220
  ✅ Item-Item Similarity only           +0.0516
  ✅ Item-Based CF only                  +0.0346
  ✅ Full Ensemble V3                    +0.0566
  ✅ Two-Stage Pipeline                  +0.0835
  ✅ Masked Basket Transformer           +0.0760

Two-Stage vs Flat Ensemble:
  Two-Stage Pipeline : 0.7276
  Full Ensemble V3  

#Cell 14b - Masked Basket Transformer

In [73]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 14b – Masked Basket Transformer (Updated V3)
# Improvements added:
#   - #3: Multi-mask training (3x more training signal)
#   - #2: Context-aware inference (meal period + weekend tokens)
# ══════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ── 14b.1  Vocabulary ─────────────────────────────────────────────────────────
PAD_TOKEN  = "<PAD>"
MASK_TOKEN = "<MASK>"

# ── Improvement #2: Context tokens added to vocabulary ───────────────────────
# These tell the transformer WHEN the order is happening
CONTEXT_TOKENS = [
    "<BREAKFAST>", "<LUNCH>", "<DINNER>", "<LATE_NIGHT>",
    "<WEEKEND>",   "<WEEKDAY>"
]

MEAL_PERIOD_TOKEN = {
    "breakfast":  "<BREAKFAST>",
    "lunch":      "<LUNCH>",
    "dinner":     "<DINNER>",
    "late_night": "<LATE_NIGHT>",
}

vocab    = [PAD_TOKEN, MASK_TOKEN] + CONTEXT_TOKENS + ALL_ITEMS
item2idx = {item: idx for idx, item in enumerate(vocab)}
idx2item = {idx: item for item, idx in item2idx.items()}

VOCAB_SIZE = len(vocab)
PAD_IDX    = item2idx[PAD_TOKEN]
MASK_IDX   = item2idx[MASK_TOKEN]

MAX_SEQ_LEN = 15   # captures more basket context

print(f"Vocabulary size : {VOCAB_SIZE}  (added {len(CONTEXT_TOKENS)} context tokens)")

# ── 14b.2  Dataset ────────────────────────────────────────────────────────────
# ── Improvement #3: Multi-mask dataset ───────────────────────────────────────
class MultiMaskBasketDataset(Dataset):
    """
    Multi-mask training: generates ONE sample per item position per basket.

    Old (single-mask): basket [A, B, C] → 1 sample
      [A, MASK, C] → B   (one random position)

    New (multi-mask): basket [A, B, C] → 3 samples
      [MASK, B, C] → A
      [A, MASK, C] → B
      [A, B, MASK] → C

    Benefits:
      - 3x more training signal from same data
      - Model learns from every position, not just random ones
      - Stronger item representations
      - Better generalization across basket sizes
    """
    def __init__(self, baskets: list, max_len: int = MAX_SEQ_LEN):
        self.samples = []
        for basket in baskets:
            basket = [it for it in basket if it in item2idx]
            if len(basket) < 2:
                continue
            basket = basket[:max_len]
            # One sample per position — NOT one random position
            for mask_pos in range(len(basket)):
                target      = item2idx[basket[mask_pos]]
                inp         = basket.copy()
                inp[mask_pos] = MASK_TOKEN
                inp_idx     = [item2idx[it] for it in inp]
                inp_idx    += [PAD_IDX] * (max_len - len(inp_idx))
                self.samples.append((inp_idx, target, mask_pos))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        inp_idx, target, mask_pos = self.samples[idx]
        return (
            torch.tensor(inp_idx,  dtype=torch.long),
            torch.tensor(target,   dtype=torch.long),
            torch.tensor(mask_pos, dtype=torch.long),
        )

# ── Build train and validation datasets ───────────────────────────────────────
train_basket_lists = train_baskets_tr["item_list"].tolist()
val_basket_lists   = val_baskets["item_list"].tolist()

train_dataset = MultiMaskBasketDataset(train_basket_lists, max_len=MAX_SEQ_LEN)
val_dataset   = MultiMaskBasketDataset(val_basket_lists,   max_len=MAX_SEQ_LEN)

train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=256, shuffle=False)

print(f"Training samples   : {len(train_dataset):,}  (was ~53k, now ~3x more)")
print(f"Validation samples : {len(val_dataset):,}")

# ── 14b.3  Model ──────────────────────────────────────────────────────────────
class MaskedBasketTransformer(nn.Module):
    """
    BERT-style transformer for basket item prediction.
    Architecture:
      Embedding → Positional Encoding → 2× TransformerEncoderLayer → Linear head

    V3 vs V1:
      - embed_dim : 64  → 128   (richer item representations)
      - ff_dim    : 128 → 256   (more expressive feed-forward)
      - num_layers: 1   → 2     (deeper = better pattern capture)
      - vocab_size: 52  → 58    (added 6 context tokens)
      - dropout   : 0.1         (unchanged)
    """
    def __init__(
        self,
        vocab_size,
        embed_dim  = 128,
        n_heads    = 4,
        ff_dim     = 256,
        num_layers = 2,
        max_len    = MAX_SEQ_LEN,
        dropout    = 0.1,
    ):
        super().__init__()
        self.embed     = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_embed = nn.Embedding(max_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = embed_dim,
            nhead           = n_heads,
            dim_feedforward = ff_dim,
            dropout         = dropout,
            batch_first     = True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(embed_dim)
        self.head        = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        positions    = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        padding_mask = (x == PAD_IDX)
        emb = self.embed(x) + self.pos_embed(positions)
        out = self.transformer(emb, src_key_padding_mask=padding_mask)
        out = self.norm(out)
        return self.head(out)   # (B, S, V)


# ── 14b.4  Training ───────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 15

model     = MaskedBasketTransformer(vocab_size=VOCAB_SIZE).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

print(f"\nTraining on {DEVICE} for {EPOCHS} epochs ...")
print(f"Model params : {sum(p.numel() for p in model.parameters()):,}")
print("-" * 55)

best_val_loss    = float("inf")
best_model_state = None
patience_counter = 0
EARLY_STOP_PATIENCE = 4

for epoch in range(1, EPOCHS + 1):

    # ── Training pass ──────────────────────────────────────────────────────────
    model.train()
    total_train_loss = 0.0

    for inp, target, mask_pos in train_loader:
        inp, target, mask_pos = (
            inp.to(DEVICE), target.to(DEVICE), mask_pos.to(DEVICE)
        )
        logits      = model(inp)
        batch_idx   = torch.arange(inp.size(0), device=DEVICE)
        mask_logits = logits[batch_idx, mask_pos, :]

        loss = criterion(mask_logits, target)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    avg_train = total_train_loss / len(train_loader)

    # ── Validation pass ────────────────────────────────────────────────────────
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for inp, target, mask_pos in val_loader:
            inp, target, mask_pos = (
                inp.to(DEVICE), target.to(DEVICE), mask_pos.to(DEVICE)
            )
            logits      = model(inp)
            batch_idx   = torch.arange(inp.size(0), device=DEVICE)
            mask_logits = logits[batch_idx, mask_pos, :]
            loss        = criterion(mask_logits, target)
            total_val_loss += loss.item()

    avg_val = total_val_loss / max(len(val_loader), 1)
    scheduler.step(avg_val)

    # ── Overfitting detection ──────────────────────────────────────────────────
    gap    = avg_val - avg_train
    status = "  ⚠️  possible overfit" if gap > 0.3 else ""

    print(
        f"  Epoch {epoch:02d}/{EPOCHS} — "
        f"Train Loss: {avg_train:.4f} | "
        f"Val Loss: {avg_val:.4f} | "
        f"Gap: {gap:+.4f}{status}"
    )

    # ── Save best model ────────────────────────────────────────────────────────
    if avg_val < best_val_loss:
        best_val_loss    = avg_val
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n  ⏹️  Early stopping at epoch {epoch} — val loss not improving.")
            break

# ── Restore best model ─────────────────────────────────────────────────────────
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✅ Restored best model (val loss = {best_val_loss:.4f})")

model.eval()
print("\n✅ Transformer training complete.")

# ── 14b.5  Inference helper ───────────────────────────────────────────────────
# ── Improvement #2: Context-aware inference ───────────────────────────────────
def transformer_scores(
    basket:      list,
    meal_period: str = "lunch",
    is_weekend:  int = 0,
) -> dict:
    """
    Context-aware inference: prepends meal period + weekend token to sequence.

    Why this helps:
      - Model knows if it's breakfast vs dinner → different item distributions
      - Model knows if it's weekend vs weekday → different ordering patterns
      - Empty basket still gives meaningful predictions via context tokens alone

    Sequence structure:
      [<LUNCH>, <WEEKDAY>, WRAP001, SAUCE001, <MASK>]
       ↑ context prefix    ↑ basket items      ↑ predict next
    """
    scores = empty_scores()

    # Build context prefix
    period_tok  = MEAL_PERIOD_TOKEN.get(meal_period, "<LUNCH>")
    weekend_tok = "<WEEKEND>" if is_weekend else "<WEEKDAY>"

    known = [it for it in basket if it in item2idx]

    # Build sequence: context tokens + basket items + MASK
    seq = [period_tok, weekend_tok] + known + [MASK_TOKEN]
    seq = seq[-MAX_SEQ_LEN:]           # keep last MAX_SEQ_LEN tokens
    mask_pos_in_seq = len(seq) - 1     # <MASK> is always last

    inp_idx  = [item2idx[it] for it in seq]
    inp_idx += [PAD_IDX] * (MAX_SEQ_LEN - len(inp_idx))

    inp_tensor = torch.tensor([inp_idx], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits = model(inp_tensor)
        probs  = torch.softmax(
            logits[0, mask_pos_in_seq, :], dim=-1
        ).cpu().numpy()

    for item in ALL_ITEMS:
        if item in item2idx:
            scores[item] = float(probs[item2idx[item]])

    return scores


# ── 14b.6  Standalone evaluation ──────────────────────────────────────────────
print("\nEvaluating Transformer signal on validation sample ...")

def transformer_only(user_id, basket, meal_period, is_weekend, k=10):
    # Now passes meal_period and is_weekend to transformer
    s        = normalize_softmax(
        transformer_scores(basket, meal_period, is_weekend)
    )
    s        = remove_basket_items(s, basket)
    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
    return top_k_items(s, k=k, fallback=fallback)

t_ndcg = run_eval("Masked Basket Transformer V3", transformer_only)
print(f"\nTransformer standalone NDCG@10 : {t_ndcg:.4f}")

print("\n✅ Masked Basket Transformer V3 complete.")

Vocabulary size : 58  (added 6 context tokens)
Training samples   : 129,942  (was ~53k, now ~3x more)
Validation samples : 14,464

Training on cuda for 15 epochs ...
Model params : 282,042
-------------------------------------------------------
  Epoch 01/15 — Train Loss: 2.1510 | Val Loss: 2.1015 | Gap: -0.0495
  Epoch 02/15 — Train Loss: 2.0962 | Val Loss: 2.0904 | Gap: -0.0058
  Epoch 03/15 — Train Loss: 2.0870 | Val Loss: 2.0908 | Gap: +0.0038
  Epoch 04/15 — Train Loss: 2.0821 | Val Loss: 2.0809 | Gap: -0.0012
  Epoch 05/15 — Train Loss: 2.0783 | Val Loss: 2.0819 | Gap: +0.0036
  Epoch 06/15 — Train Loss: 2.0767 | Val Loss: 2.0809 | Gap: +0.0042
  Epoch 07/15 — Train Loss: 2.0748 | Val Loss: 2.0769 | Gap: +0.0021
  Epoch 08/15 — Train Loss: 2.0738 | Val Loss: 2.0829 | Gap: +0.0091
  Epoch 09/15 — Train Loss: 2.0721 | Val Loss: 2.0822 | Gap: +0.0101
  Epoch 10/15 — Train Loss: 2.0712 | Val Loss: 2.0773 | Gap: +0.0061
  Epoch 11/15 — Train Loss: 2.0595 | Val Loss: 2.0723 | Gap: +0.0

# Cell 15 Final Prediction Pipeline




In [75]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 15 – Final Prediction Pipeline (Updated V3)
# Improvements:
#   - Uses MultiMaskBasketDataset for full retrain (3x more signal)
#   - Context-aware transformer_scores (meal_period + is_weekend)
#   - Uses two_stage_recommend as primary prediction function
# ══════════════════════════════════════════════════════════════════════════════

print("Step 1/5 – Rebuilding matrices on full training data ...")

# ── Full weighted matrix ───────────────────────────────────────────────────────
agg_full = (
    train_transactions
    .groupby(["user_id", "item_id"])
    .agg(
        freq        = ("item_id",        "count"),
        recency_sum = ("recency_weight",  "sum"),
    )
    .reset_index()
)

wt_matrix_full   = agg_full.pivot(index="user_id", columns="item_id", values="recency_sum").fillna(0)
freq_matrix_full = agg_full.pivot(index="user_id", columns="item_id", values="freq").fillna(0)

for item in ALL_ITEMS:
    if item not in wt_matrix_full.columns:
        wt_matrix_full[item] = 0.0
    if item not in freq_matrix_full.columns:
        freq_matrix_full[item] = 0.0

wt_matrix_full   = wt_matrix_full[ALL_ITEMS]
freq_matrix_full = freq_matrix_full[ALL_ITEMS]
binary_full      = (freq_matrix_full > 0).astype(float)

print(f"  Full wt_matrix shape : {wt_matrix_full.shape}")

# ── Step 2: Rebuild item-item similarity ──────────────────────────────────────
print("Step 2/5 – Rebuilding item-item similarity on full data ...")

item_vectors_full = wt_matrix_full.T.values
item_sim_full     = cosine_similarity(item_vectors_full)
item_ids_full     = wt_matrix_full.columns.tolist()
item_idx_full     = {item: i for i, item in enumerate(item_ids_full)}

print(f"  Item-sim matrix shape : {item_sim_full.shape}")

# ── Step 3: Rebuild context popularity ────────────────────────────────────────
print("Step 3/5 – Rebuilding context popularity on full data ...")

tx_full = train_transactions.copy()
tx_full["category"] = tx_full["item_id"].map(
    lambda x: item_meta.get(x, {}).get("category", "Unknown")
)

ctx_pop_full = {}
for period, grp in tx_full.groupby("meal_period"):
    counts = grp["item_id"].value_counts()
    ctx_pop_full[period] = (counts / counts.sum()).to_dict()

wknd_pop_full = {}
for is_wknd, grp in tx_full.groupby("is_weekend"):
    counts = grp["item_id"].value_counts()
    wknd_pop_full[is_wknd] = (counts / counts.sum()).to_dict()

global_pop_full = tx_full["item_id"].value_counts()
global_pop_full = (global_pop_full / global_pop_full.sum()).to_dict()

print(f"  Context periods found : {list(ctx_pop_full.keys())}")

# ── Step 4: Retrain Transformer on FULL data ──────────────────────────────────
print("Step 4/5 – Retraining Transformer on full basket data ...")

# Use ALL baskets (train + val) — no holdout needed for final model
all_basket_lists = train_baskets["item_list"].tolist()

# ── Use MultiMaskBasketDataset (V3 improvement #3) ───────────────────────────
full_trans_dataset = MultiMaskBasketDataset(all_basket_lists, max_len=MAX_SEQ_LEN)
full_trans_loader  = DataLoader(full_trans_dataset, batch_size=256, shuffle=True)

print(f"  Full transformer training samples : {len(full_trans_dataset):,}  (~3x more than V2)")

# Reinitialize model with same V3 architecture
model_final     = MaskedBasketTransformer(vocab_size=VOCAB_SIZE).to(DEVICE)
optimizer_final = optim.Adam(model_final.parameters(), lr=1e-3)
criterion_final = nn.CrossEntropyLoss(label_smoothing=0.1)

# Use best epoch count from Cell 14b early stopping
FINAL_EPOCHS = EPOCHS   # inherits from Cell 14b

model_final.train()
for epoch in range(1, FINAL_EPOCHS + 1):
    total_loss = 0.0
    for inp, target, mask_pos in full_trans_loader:
        inp, target, mask_pos = (
            inp.to(DEVICE), target.to(DEVICE), mask_pos.to(DEVICE)
        )
        logits      = model_final(inp)
        batch_idx   = torch.arange(inp.size(0), device=DEVICE)
        mask_logits = logits[batch_idx, mask_pos, :]
        loss        = criterion_final(mask_logits, target)
        optimizer_final.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_final.parameters(), max_norm=1.0)
        optimizer_final.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(full_trans_loader)
    print(f"  Epoch {epoch:02d}/{FINAL_EPOCHS} — Loss: {avg_loss:.4f}")

model_final.eval()
print("  ✅ Final transformer trained on full data.")

# ── Override all in-memory objects with full-data versions ────────────────────
wt_matrix          = wt_matrix_full
item_sim_matrix    = item_sim_full
item_ids_ordered   = item_ids_full
item_idx           = item_idx_full
context_popularity = ctx_pop_full
weekend_popularity = wknd_pop_full
global_popularity  = global_pop_full

# ── Override transformer_scores with context-aware full-data version ──────────
# Improvement #2: passes meal_period + is_weekend to model
def transformer_scores(
    basket:      list,
    meal_period: str = "lunch",
    is_weekend:  int = 0,
) -> dict:
    """
    Context-aware inference using the FULL-DATA trained transformer.
    Overrides Cell 14b version for final predictions.

    Sequence: [<MEAL_PERIOD>, <WEEKEND/WEEKDAY>, item1, item2, ..., <MASK>]
    Empty basket → context tokens + MASK → still gives useful distribution.
    """
    scores = empty_scores()

    # Build context prefix
    period_tok  = MEAL_PERIOD_TOKEN.get(meal_period, "<LUNCH>")
    weekend_tok = "<WEEKEND>" if is_weekend else "<WEEKDAY>"

    known = [it for it in basket if it in item2idx]

    # Sequence: context prefix + basket items + MASK
    seq = [period_tok, weekend_tok] + known + [MASK_TOKEN]
    seq = seq[-MAX_SEQ_LEN:]
    mask_pos_in_seq = len(seq) - 1

    inp_idx  = [item2idx[it] for it in seq]
    inp_idx += [PAD_IDX] * (MAX_SEQ_LEN - len(inp_idx))

    inp_tensor = torch.tensor([inp_idx], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits = model_final(inp_tensor)
        probs  = torch.softmax(
            logits[0, mask_pos_in_seq, :], dim=-1
        ).cpu().numpy()

    for item in ALL_ITEMS:
        if item in item2idx:
            scores[item] = float(probs[item2idx[item]])

    return scores

print("\nStep 5/5 – Generating predictions for test instances ...")

submission_rows = []

for i, row in test_instances.iterrows():
    instance_id = row["instance_id"]
    user_id     = str(row["user_id"])
    basket      = row["basket_list"]
    meal_period = row["meal_period"]
    is_weekend  = int(row["is_weekend"])

    # ── Use two_stage_recommend as primary function (V3) ──────────────────────
    recs = two_stage_recommend(
        user_id      = user_id,
        basket       = basket,
        meal_period  = meal_period,
        is_weekend   = is_weekend,
        k            = 10,
        n_candidates = 20,
    )

    submission_rows.append({
        "instance_id": instance_id,
        **{f"rank_{j+1}": recs[j] for j in range(10)},
    })

    if (i + 1) % 2000 == 0:
        print(f"  Processed {i+1:,} / {len(test_instances):,} instances ...")

submission_df = pd.DataFrame(submission_rows)

print(f"\nSubmission shape : {submission_df.shape}")
print("\nSample rows:")
print(submission_df.head(3).to_string())
print("\n✅ Predictions generated.")

Step 1/5 – Rebuilding matrices on full training data ...
  Full wt_matrix shape : (9054, 50)
Step 2/5 – Rebuilding item-item similarity on full data ...
  Item-sim matrix shape : (50, 50)
Step 3/5 – Rebuilding context popularity on full data ...
  Context periods found : ['breakfast', 'dinner', 'late_night', 'lunch']
Step 4/5 – Retraining Transformer on full basket data ...
  Full transformer training samples : 144,406  (~3x more than V2)
  Epoch 01/15 — Loss: 2.1504
  Epoch 02/15 — Loss: 2.0956
  Epoch 03/15 — Loss: 2.0849
  Epoch 04/15 — Loss: 2.0821
  Epoch 05/15 — Loss: 2.0797
  Epoch 06/15 — Loss: 2.0777
  Epoch 07/15 — Loss: 2.0748
  Epoch 08/15 — Loss: 2.0745
  Epoch 09/15 — Loss: 2.0722
  Epoch 10/15 — Loss: 2.0726
  Epoch 11/15 — Loss: 2.0709
  Epoch 12/15 — Loss: 2.0692
  Epoch 13/15 — Loss: 2.0678
  Epoch 14/15 — Loss: 2.0692
  Epoch 15/15 — Loss: 2.0665
  ✅ Final transformer trained on full data.

Step 5/5 – Generating predictions for test instances ...
  Processed 2,000 / 

# Cell 16 - Submission File Generation ..



In [76]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 16 – Submission File Generation
# ══════════════════════════════════════════════════════════════════════════════

# ── 16.1  Validate format against sample_submission.csv ──────────────────────
expected_cols = list(sample_submission.columns)
actual_cols   = list(submission_df.columns)

assert actual_cols == expected_cols, (
    f"Column mismatch!\nExpected: {expected_cols}\nGot: {actual_cols}"
)

# ── 16.2  Check all instance_ids are covered ─────────────────────────────────
expected_ids = set(sample_submission["instance_id"])
actual_ids   = set(submission_df["instance_id"])

missing = expected_ids - actual_ids
extra   = actual_ids   - expected_ids

print(f"Expected instances  : {len(expected_ids):,}")
print(f"Submitted instances : {len(actual_ids):,}")
print(f"Missing             : {len(missing)}")
print(f"Extra               : {len(extra)}")

assert len(missing) == 0, f"Missing {len(missing)} instances!"

# ── 16.3  Validate item IDs are all from menu ─────────────────────────────────
rank_cols    = [f"rank_{i}" for i in range(1, 11)]
all_pred_ids = submission_df[rank_cols].values.flatten()
invalid_ids  = set(all_pred_ids) - ITEM_SET

print(f"Invalid item IDs in predictions : {len(invalid_ids)}")
assert len(invalid_ids) == 0, f"Found invalid items: {invalid_ids}"

# ── 16.4  Ensure no duplicates within a row ───────────────────────────────────
def check_row_uniqueness(row):
    items = row[rank_cols].tolist()
    return len(items) == len(set(items))

all_unique = submission_df.apply(check_row_uniqueness, axis=1).all()
print(f"All rows have unique items      : {all_unique}")

# ── 16.5  Reorder rows to match sample_submission order ──────────────────────
submission_df = submission_df.set_index("instance_id")
submission_df = submission_df.loc[sample_submission["instance_id"]]
submission_df = submission_df.reset_index()

# ── 16.6  Save ───────────────────────────────────────────────────────────────
OUTPUT_FILE = "submission.csv"
submission_df.to_csv(OUTPUT_FILE, index=False)

print(f"\n💾 Submission saved to: {OUTPUT_FILE}")
print(f"   Rows : {len(submission_df):,}")
print(f"   Cols : {list(submission_df.columns)}")

print("\nFinal preview:")
print(submission_df.head(5).to_string())

print("\n🎉 Done! Upload submission.csv to the leaderboard.")

Expected instances  : 14,147
Submitted instances : 14,147
Missing             : 0
Extra               : 0
Invalid item IDs in predictions : 0
All rows have unique items      : True

💾 Submission saved to: submission.csv
   Rows : 14,147
   Cols : ['instance_id', 'rank_1', 'rank_2', 'rank_3', 'rank_4', 'rank_5', 'rank_6', 'rank_7', 'rank_8', 'rank_9', 'rank_10']

Final preview:
   instance_id    rank_1   rank_2    rank_3  rank_4    rank_5   rank_6   rank_7    rank_8   rank_9   rank_10
0  test_001579   WRAP001  SIDE001  SAUCE001  BEV006   HIGH001  WRAP004  WRAP002  PLATE001  WRAP006    BEV001
1  test_000767    BEV006  SIDE001   WRAP001  BEV001  SAUCE003   BEV007   BEV002  PLATE001   BEV003   HIGH001
2  test_000266  SAUCE001   BEV006   SIDE001  BEV001    BEV007   APP006  WRAP001    APP004   APP001    APP005
3  test_005435   SIDE001   BEV006  SAUCE003  BEV001   WRAP001   BEV007   BEV002    BEV003   BEV004  PLATE001
4  test_011299    BEV006  SIDE001   WRAP001  BEV001  SAUCE003   BEV007   BE

In [77]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>